## Data Cleaning


In [2]:
from datasets import load_dataset
import pandas as pd
import re

/Users/ashtondy/miniconda3/envs/cs424-proj2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset = load_dataset("Ahmad0067/MedSynth")
data = dataset["train"]

print(len(data))
print(data[0])

Generating train split: 100%|██████████| 10240/10240 [00:00<00:00, 19590.37 examples/s]

10240
{' Note': "**1. Subjective:**\n\n   **Chief Complaint (CC):**\n   - Pain in the left knee, moderate to severe, lasting for 3 weeks.\n\n   **History of Present Illness (HPI):**\n   - The patient, a 52-year-old Caucasian male named John Smith, presents with complaints of moderate to severe pain in the left knee that has persisted for the past three weeks. The pain is associated with occasional swelling and stiffness, particularly pronounced in the mornings. The symptoms are exacerbated by physical activity and have a significant impact on daily activities, including walking, climbing stairs, and prolonged standing.\n\n   **Review of Systems (ROS):**\n   - Musculoskeletal: Positive for knee pain, swelling, and stiffness.\n   - General: Negative for fever or weight loss.\n   - Cardiovascular: Negative for chest pain or palpitations.\n   - Constitutional: Sleep disruption due to knee pain; otherwise stable.\n\n**2. Objective:**\n\n   **Vital Signs:**\n   - Blood Pressure: 128/82 mmHg\

In [4]:
for i in range(3):
    print("---- SAMPLE ----")
    print(data[i]["Dialogue"])
    print("\nSOAP:", data[i][" Note"])
    print("\n")

---- SAMPLE ----
[doctor]: Hello! It’s good to see you today. How can I help you?

[patient]: Hi, Doctor. I’ve been having a lot of pain in my left knee.

[doctor]: I’m sorry to hear that. Can you tell me more about the pain and how long it’s been going on?

[patient]: Sure. It's been pretty bad, moderate to severe pain, for the last three weeks.

[doctor]: Hmm, that sounds uncomfortable. Is there anything specific that makes the pain worse?

[patient]: Yes, physical activities really make it worse. Walking, climbing stairs, standing for long periods—it all makes it more painful.

[doctor]: I see. Have you noticed any swelling or stiffness in the knee?

[patient]: Yes, there’s occasional swelling and it’s really stiff, especially in the mornings.

[doctor]: Alright, any other symptoms you’ve noticed? Fever, weight loss, anything like that?

[patient]: No, no fever or weight loss.

[doctor]: How about chest pain or palpitations? Anything unusual with your heart?

[patient]: No, my heart

In [5]:
import re

def clean_transcript(text):
    if not text:
        return None

    text = text.strip()
    text = re.sub(r'\s+', ' ', text)

    text = text.replace("Doctor:", "[Doctor]:")
    text = text.replace("Patient:", "[Patient]:")

    MAX_CHARS = 4000
    return text[:MAX_CHARS]


def clean_soap_note(text):
    if not text:
        return None

    text = text.replace("**", "")
    text = text.replace("##", "")

    text = text.strip()
    text = re.sub(r'\s+', ' ', text)

    return text

In [6]:
sample = data[0]

raw_transcript = sample["Dialogue"]
raw_soap = sample[" Note"]

clean_trans = clean_transcript(raw_transcript)
clean_soap = clean_soap_note(raw_soap)

print("===== RAW TRANSCRIPT =====\n")
print(raw_transcript)

print("\n===== CLEAN TRANSCRIPT =====\n")
print(clean_trans)

print("\n============================\n")

print("===== RAW SOAP =====\n")
print(raw_soap)

print("\n===== CLEAN SOAP =====\n")
print(clean_soap)

===== RAW TRANSCRIPT =====

[doctor]: Hello! It’s good to see you today. How can I help you?

[patient]: Hi, Doctor. I’ve been having a lot of pain in my left knee.

[doctor]: I’m sorry to hear that. Can you tell me more about the pain and how long it’s been going on?

[patient]: Sure. It's been pretty bad, moderate to severe pain, for the last three weeks.

[doctor]: Hmm, that sounds uncomfortable. Is there anything specific that makes the pain worse?

[patient]: Yes, physical activities really make it worse. Walking, climbing stairs, standing for long periods—it all makes it more painful.

[doctor]: I see. Have you noticed any swelling or stiffness in the knee?

[patient]: Yes, there’s occasional swelling and it’s really stiff, especially in the mornings.

[doctor]: Alright, any other symptoms you’ve noticed? Fever, weight loss, anything like that?

[patient]: No, no fever or weight loss.

[doctor]: How about chest pain or palpitations? Anything unusual with your heart?

[patient]: N

In [7]:
def clean_transcript(text):
    if not text:
        return None

    # normalize spaces BUT keep line breaks
    text = text.strip()

    # standardize speakers
    text = text.replace("[doctor]:", "[Doctor]:")
    text = text.replace("[patient]:", "[Patient]:")

    return text


def clean_soap_note(text):
    if not text:
        return None

    # remove markdown symbols
    text = text.replace("**", "")
    text = text.replace("##", "")

    # remove only section numbers like "1. ", "2. "
    text = re.sub(r'^\d+\.\s*', '', text, flags=re.MULTILINE)

    # normalize spacing but KEEP line breaks
    text = text.strip()

    return text

In [8]:
sample = data[0]

raw_transcript = sample["Dialogue"]
raw_soap = sample[" Note"]

clean_trans = clean_transcript(raw_transcript)
clean_soap = clean_soap_note(raw_soap)

print("===== RAW TRANSCRIPT =====\n")
print(raw_transcript)

print("\n===== CLEAN TRANSCRIPT =====\n")
print(clean_trans)

print("\n============================\n")

print("===== RAW SOAP =====\n")
print(raw_soap)

print("\n===== CLEAN SOAP =====\n")
print(clean_soap)

===== RAW TRANSCRIPT =====

[doctor]: Hello! It’s good to see you today. How can I help you?

[patient]: Hi, Doctor. I’ve been having a lot of pain in my left knee.

[doctor]: I’m sorry to hear that. Can you tell me more about the pain and how long it’s been going on?

[patient]: Sure. It's been pretty bad, moderate to severe pain, for the last three weeks.

[doctor]: Hmm, that sounds uncomfortable. Is there anything specific that makes the pain worse?

[patient]: Yes, physical activities really make it worse. Walking, climbing stairs, standing for long periods—it all makes it more painful.

[doctor]: I see. Have you noticed any swelling or stiffness in the knee?

[patient]: Yes, there’s occasional swelling and it’s really stiff, especially in the mornings.

[doctor]: Alright, any other symptoms you’ve noticed? Fever, weight loss, anything like that?

[patient]: No, no fever or weight loss.

[doctor]: How about chest pain or palpitations? Anything unusual with your heart?

[patient]: N

# Check for NULL / Missing Values

In [9]:
cleaned_data = []

for sample in data:
    transcript = clean_transcript(sample["Dialogue"])
    soap = clean_soap_note(sample[" Note"])  

    if not transcript or not soap:
        continue

    cleaned_data.append({
        "transcript": transcript,
        "ground_truth": soap
    })

print("Total cleaned samples:", len(cleaned_data))

Total cleaned samples: 10238


In [10]:
import pandas as pd

df = pd.DataFrame(cleaned_data)
print(df.head())

                                          transcript  \
0  [Doctor]: Hello! It’s good to see you today. H...   
1  [doctor] Hi there, how are you today?\n\n[pati...   
2  [doctor] Good morning, how are you doing today...   
3  [doctor] Good morning! How are you feeling tod...   
4  [Doctor]: Hello Mr. Doe, how are you doing tod...   

                                        ground_truth  
0  Subjective:\n\n   Chief Complaint (CC):\n   - ...  
1  Subjective:\n\n   - Chief Complaint (CC): Pain...  
2  Subjective:\n\nChief Complaint (CC):\nSevere r...  
3  Subjective:\n\nChief Complaint (CC):  \nModera...  
4  #\nSubjective:\n\nChief Complaint (CC):  \nMod...  


In [11]:
empty_transcripts = df[df["transcript"].str.strip() == ""]
empty_soap = df[df["ground_truth"].str.strip() == ""]

print("Empty transcripts:", len(empty_transcripts))
print("Empty SOAP:", len(empty_soap))

Empty transcripts: 0
Empty SOAP: 0


In [12]:
print("Total duplicate rows:", df.duplicated().sum())

Total duplicate rows: 205


In [13]:
dup_transcripts = df.duplicated(subset=["transcript"]).sum()
print("Duplicate transcripts:", dup_transcripts)

Duplicate transcripts: 205


In [14]:
df = df.drop_duplicates(subset=["transcript"])

print("After removing duplicates:", len(df))

After removing duplicates: 10033


In [15]:
df["length"] = df["transcript"].apply(len)

print(df["length"].describe())

count    10033.000000
mean      4537.760291
std        657.862512
min       1897.000000
25%       4089.000000
50%       4504.000000
75%       4943.000000
max      17453.000000
Name: length, dtype: float64


In [16]:
data = data.rename_column(" Note", "Note")

In [17]:
for i in range(3):
    print("====== SAMPLE", i, "======\n")
    
    print("TRANSCRIPT:\n")
    print(df.iloc[i]["transcript"])
    
    print("\nSOAP:\n")
    print(df.iloc[i]["ground_truth"])
    
    print("\n" + "="*50 + "\n")

====== SAMPLE 0 ======

TRANSCRIPT:

[Doctor]: Hello! It’s good to see you today. How can I help you?

[Patient]: Hi, Doctor. I’ve been having a lot of pain in my left knee.

[Doctor]: I’m sorry to hear that. Can you tell me more about the pain and how long it’s been going on?

[Patient]: Sure. It's been pretty bad, moderate to severe pain, for the last three weeks.

[Doctor]: Hmm, that sounds uncomfortable. Is there anything specific that makes the pain worse?

[Patient]: Yes, physical activities really make it worse. Walking, climbing stairs, standing for long periods—it all makes it more painful.

[Doctor]: I see. Have you noticed any swelling or stiffness in the knee?

[Patient]: Yes, there’s occasional swelling and it’s really stiff, especially in the mornings.

[Doctor]: Alright, any other symptoms you’ve noticed? Fever, weight loss, anything like that?

[Patient]: No, no fever or weight loss.

[Doctor]: How about chest pain or palpitations? Anything unusual with your heart?

[Pa

In [18]:
import random

for _ in range(3):
    idx = random.randint(0, len(df)-1)
    
    print("====== RANDOM SAMPLE ======\n")
    
    print("TRANSCRIPT:\n")
    print(df.iloc[idx]["transcript"][:500])  # preview
    
    print("\nSOAP:\n")
    print(df.iloc[idx]["ground_truth"][:500])
    
    print("\n" + "="*50 + "\n")

====== RANDOM SAMPLE ======

TRANSCRIPT:

[Doctor]: Hi there, John. How are you feeling today?

[Patient]: Hi, Doctor. Not great, to be honest.

[Doctor]: I see. What brings you in today?

[Patient]: I've been having a really bad cough for the past five days. It's been productive, with yellow sputum. I also have a fever and feel extremely tired. The worst part is the chest pain when I take deep breaths.

[Doctor]: That sounds quite uncomfortable. How high has your fever been?

[Patient]: The highest it got was 101°F.

[Doctor]: And you 

SOAP:

Subjective:

Chief Complaint (CC):
Productive cough, fever, shortness of breath, and chest pain on deep inspiration.

History of Present Illness (HPI):
John Smith, a 45-year-old Caucasian male, presents with a 5-day history of productive cough, fever, shortness of breath, and chest pain upon deep inspiration. The cough is productive of yellow sputum and is continuous throughout the day, adversely affecting his sleep quality. He reports a fever p

In [19]:
def clean_transcript(text):
    if not text:
        return None

    text = text.strip()

    # normalize all variations
    text = text.replace("[doctor]", "[Doctor]")
    text = text.replace("[patient]", "[Patient]")

    text = text.replace("[doctor]:", "[Doctor]:")
    text = text.replace("[patient]:", "[Patient]:")

    return text

In [20]:
def normalize_sections(text):
    text = re.sub(r'#\s*SUBJECTIVE', 'Subjective:', text, flags=re.IGNORECASE)
    text = re.sub(r'#\s*OBJECTIVE', 'Objective:', text, flags=re.IGNORECASE)
    text = re.sub(r'#\s*ASSESSMENT', 'Assessment:', text, flags=re.IGNORECASE)
    text = re.sub(r'#\s*PLAN', 'Plan:', text, flags=re.IGNORECASE)

    text = re.sub(r'#\s*\d+\.\s*Subjective:', 'Subjective:', text, flags=re.IGNORECASE)
    text = re.sub(r'#\s*\d+\.\s*Objective:', 'Objective:', text, flags=re.IGNORECASE)

    return text

In [21]:
def clean_soap_note(text):
    if not text:
        return None

    text = text.replace("**", "")
    text = text.replace("##", "")

    text = text.replace("\\n", "\n")

    text = re.sub(r'^\d+\.\s*', '', text, flags=re.MULTILINE)

    text = normalize_sections(text)

    return text.strip()

In [22]:
for i in range(2):
    print("="*60)
    print(f"SAMPLE {i}")
    print("="*60)
    
    print("\n--- TRANSCRIPT ---\n")
    print(df.iloc[i]["transcript"])
    
    print("\n--- SOAP NOTE ---\n")
    print(df.iloc[i]["ground_truth"])
    
    print("\n\n")

SAMPLE 0

--- TRANSCRIPT ---

[Doctor]: Hello! It’s good to see you today. How can I help you?

[Patient]: Hi, Doctor. I’ve been having a lot of pain in my left knee.

[Doctor]: I’m sorry to hear that. Can you tell me more about the pain and how long it’s been going on?

[Patient]: Sure. It's been pretty bad, moderate to severe pain, for the last three weeks.

[Doctor]: Hmm, that sounds uncomfortable. Is there anything specific that makes the pain worse?

[Patient]: Yes, physical activities really make it worse. Walking, climbing stairs, standing for long periods—it all makes it more painful.

[Doctor]: I see. Have you noticed any swelling or stiffness in the knee?

[Patient]: Yes, there’s occasional swelling and it’s really stiff, especially in the mornings.

[Doctor]: Alright, any other symptoms you’ve noticed? Fever, weight loss, anything like that?

[Patient]: No, no fever or weight loss.

[Doctor]: How about chest pain or palpitations? Anything unusual with your heart?

[Patient]:

In [23]:
import random

for _ in range(2):
    idx = random.randint(0, len(df)-1)
    
    print("="*60)
    print(f"RANDOM SAMPLE {idx}")
    print("="*60)
    
    print("\n--- TRANSCRIPT ---\n")
    print(df.iloc[idx]["transcript"])
    
    print("\n--- SOAP NOTE ---\n")
    print(df.iloc[idx]["ground_truth"])
    
    print("\n\n")

RANDOM SAMPLE 3053

--- TRANSCRIPT ---

[Doctor]: Hi there, how are you doing today?

[Patient]: Hi, doctor. I've been better. I'm struggling quite a bit lately.

[Doctor]: Hmm, I see. I read that you're experiencing some troubling symptoms. Can you tell me more about what's been going on?

[Patient]: Well, over the past 8 months, I've been hearing voices that aren't there, and it's really scary. I'm also finding it hard to interact with people and my speech is sometimes all over the place. I just don't enjoy things like I used to.

[Doctor]: I'm sorry to hear that. It sounds very challenging. How often are these symptoms occurring?

[Patient]: It happens several times a week, and it's really affecting my daily life and work.

[Doctor]: That sounds quite severe. How has this impacted your job and daily tasks?

[Patient]: I'm a software engineer, and it's been tough to keep up with my work. I've been missing a lot of days, and even basic daily tasks feel overwhelming. My personal hygien

In [24]:
df.to_json("clean_medsynth_final.json", orient="records", indent=2)

In [25]:
df.to_csv("clean_medsynth_final.csv", index=False)